# 02 - Landmark train + eval for ONE cutoff week (parameter: `cutoff_week`)
Single multilabel model, open horizon: P(outcome diagnosed after the landmark) among patients still pregnant and with no APO yet.
Pipeline: load eligible patients (fixed split from the prep notebook) -> train global branch -> train local branch -> freeze -> fuse
(fixed `lambda_l1`) -> held-out test metrics + bootstrap CIs -> write Delta tables -> pickle export, all keyed by `model_id = landmark_wk{kk}`.
Designed to run as one iteration of a Databricks for-each job; safe to rerun (set `force=true` to overwrite a finished cutoff).

In [0]:
import threading, time

CFG_MON = {"interval_s": 2.0}
gpu_samples, _stop = [], threading.Event()

def _sample():
    while not _stop.is_set():
        gpu_samples.append((time.time(), torch.cuda.utilization()))   # % of time a kernel was running
        time.sleep(CFG_MON["interval_s"])

threading.Thread(target=_sample, daemon=True).start()

In [0]:
import os, time, copy, pickle, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pyspark.sql.functions as sf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             brier_score_loss, balanced_accuracy_score)

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")   # fast Spark -> pandas for wide embedding frames

dbutils.widgets.text("cutoff_week", "25")
dbutils.widgets.dropdown("force", "false", ["false", "true"])
CUTOFF_WEEK = int(dbutils.widgets.get("cutoff_week"))
FORCE = dbutils.widgets.get("force") == "true"
kk = f"{CUTOFF_WEEK:02d}"

CLASS_NAMES = ["control", "pe_ghtn", "gdm"]
OUTCOME_NAMES = ["pe_ghtn", "gdm"]
LABEL_COLS = ["y_control", "y_pe_ghtn", "y_gdm"]

# ============================================================================
# Config -- all hyperparameters centralized here
# ============================================================================
FINAL_CFG_GLOBAL = {'flags_path': 'catalog.schema.step7_filtered_cohort_with_apos', 'output_dir': './proto_runs', 'embedding_dim': 768, 'adapter_hidden': 256, 'adapter_out': 64, 'n_classes': 3, 'prototypes_per_class': {'control': 17, 'pe_ghtn': 9, 'gdm': 9}, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 512, 'max_epochs': 200, 'patience': 10, 'lambda_clst': 0.00045845413117644694, 'lambda_sep': 0.002298951753515831, 'lambda_div': 49.337245549499, 'lambda_cntrst': 66.98658887574608, 'lambda_covar': 0.0579582453330533, 'zero_diagonal_blocks': True, 'use_jscl_pretrain': False, 'max_iterations': 5, 'iter_patience': 2, 'lr_decay': 0.5, 'random_seed': 42, 'run_id': 'global_final', '_branch_name': 'global'}
FINAL_CFG_LOCAL  = {'flags_path': 'catalog.schema.step7_filtered_cohort_with_apos', 'output_dir': './proto_runs', 'embedding_dim': 768, 'adapter_hidden': 256, 'adapter_out': 64, 'n_classes': 3, 'prototypes_per_class': {'control': 6, 'pe_ghtn': 5, 'gdm': 18}, 'lr': 0.0005, 'weight_decay': 0.0001, 'batch_size': 512, 'max_epochs': 200, 'patience': 10, 'lambda_clst': 0.007368247757287483, 'lambda_sep': 0.006551162993545934, 'lambda_div': 0.4195993256985269, 'lambda_cntrst': 65.47464665971741, 'lambda_covar': 2.658175767843088, 'zero_diagonal_blocks': True, 'use_jscl_pretrain': False, 'max_iterations': 5, 'iter_patience': 2, 'lr_decay': 0.5, 'random_seed': 42, 'run_id': 'local_final', 'topk_pool_k': 5, 'max_train_patients_per_stratum': None, '_branch_name': 'local'}

CFG = {
    "cutoff_week": CUTOFF_WEEK,
    "model_id": f"landmark_wk{kk}",
    # "model_id": "replication_v5",

    # inputs (written by the inference notebook and the prep notebook)
    "global_emb_table": f"catalog.schema.patient_clmbr_embeddings_landmark_wk{kk}",
    # "global_emb_table": "catalog.schema.patient_clmbr_embeddings_new_v5",
    "local_emb_table": f"catalog.schema.patient_clmbr_embeddings_landmark_delta_wk{kk}",
    # "local_emb_table": "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester_delta",
    "landmark_cohort_table": "catalog.schema.protopregnet_landmark_cohort",
    "eligibility_table": "catalog.schema.protopregnet_landmark_eligibility",
    "test_code_set_cache": "/path/to/exports/landmark_test_code_sets.pkl",

    # outputs
    "metrics_table": "catalog.schema.protopregnet_landmark_metrics",
    "sizes_table": "catalog.schema.protopregnet_landmark_sizes",
    "export_path": f"/path/to/exports/full_analysis_export_landmark_wk{kk}.pkl",
    # "export_path": "/path/to/exports/full_analysis_export_replication_v5.pkl",
    "write_retries": 3,

    # fusion: lambda_l1 fixed from the earlier sweep (no sweep here); the rest are train_fusion_layer's defaults
    "fusion": {"lambda_l1": 3.75341e-05, "lr": 1e-3, "weight_decay": 1e-4,
               "max_epochs": 200, "patience": 15, "batch_size": 512, "seed": 42},

    # pooling / chunking (inference-only, does not change training math)
    "topk_pool_k": FINAL_CFG_LOCAL["topk_pool_k"],
    "infer_chunk": 16384,

    # bootstrap CIs on the held-out test set (same procedure as the fusion notebook)
    "n_bootstrap": 1000, "ci": 0.95, "bootstrap_seed": 42,
}

assert torch.cuda.is_available(), "No GPU -- stopping rather than silently running on CPU."
device = torch.device("cuda")
print(f"{CFG['model_id']} | torch={torch.__version__} | GPU={torch.cuda.get_device_name(0)}")

def already_done():
    n = spark.table(CFG["metrics_table"]).filter(sf.col("model_id") == CFG["model_id"]).count()
    return n > 0 and os.path.exists(CFG["export_path"])

if already_done() and not FORCE:
    dbutils.notebook.exit(f"{CFG['model_id']} already done (metrics rows + pickle exist); rerun with force=true to overwrite")

In [0]:
# ============================================================================
# Data loading: eligible patients at this landmark, from the fixed split
# ============================================================================
def load_eligible_cohort(week):
    """People eligible at this cutoff (rule defined once, in the prep notebook), with split/labels/age."""
    elig = spark.table(CFG["eligibility_table"]).filter(sf.col("cutoff_week") == week).select("pid_int")
    return spark.table(CFG["landmark_cohort_table"]).join(sf.broadcast(elig), "pid_int").toPandas()


def pull_embeddings(table, coh, local):
    """One Spark-side join + Arrow transfer (replaces the 2000-id IN-list chunks).
    LOCAL: keep only available, non-NaN delta rows -- the same filter as load_local_data
    (ramp-up rows are all-NaN, so checking emb_0 is enough)."""
    ids = spark.createDataFrame(coh[["pid_int"]])
    sdf = spark.table(table)
    if local:
        sdf = (sdf.filter((sf.col("unavailable") == 0) & sf.col("emb_0").isNotNull() & ~sf.isnan("emb_0"))
                  .drop("n_tokens", "unavailable", "empty"))
    else:
        sdf = sdf.drop("sequence_length")
    sdf = sdf.join(sf.broadcast(ids), sdf["patient_id"] == ids["pid_int"]).drop("pid_int")
    return sdf.toPandas()


def to_split_arrays(rows, coh, branch):
    """Rows -> per-split numpy arrays (X, Y, pid, age[, window_name]) with patient labels broadcast over rows."""
    emb_cols = sorted([c for c in rows.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
    assert len(emb_cols) == FINAL_CFG_GLOBAL["embedding_dim"]
    rows = rows.merge(coh[["pid_int", "split", *LABEL_COLS, "age"]],
                      left_on="patient_id", right_on="pid_int", how="inner")
    out = {}
    for name in ["train", "val", "test"]:
        sub = rows[rows["split"] == name]
        out[f"X_{name}"] = sub[emb_cols].to_numpy(dtype=np.float32)
        out[f"Y_{name}"] = sub[LABEL_COLS].to_numpy(dtype=np.float32)
        out[f"pid_{name}"] = sub["patient_id"].astype(str).to_numpy()
        out[f"age_{name}"] = sub["age"].to_numpy(dtype=np.float32)
        if branch == "local":
            out[f"window_name_{name}"] = sub["window_name"].to_numpy()
    # class weights from THIS cutoff's training set (row-weighted for local, as in the original)
    pos_freq = out["Y_train"].mean(axis=0).clip(0.03, 0.97)
    out["class_weights"] = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float32).to(device)
    return out


def attach_gpu(data):
    """Put the whole training/val/test tensors on the GPU once: no DataLoader, no per-batch host copies."""
    for name in ["train", "val", "test"]:
        data[f"X_{name}_g"] = torch.as_tensor(data[f"X_{name}"], dtype=torch.float32, device=device)
    data["Y_train_g"] = torch.as_tensor(data["Y_train"], dtype=torch.float32, device=device)
    return data


def build_pad_index(pid):
    """Row -> (patient, position-within-patient) index so top-k pooling can run as ONE batched GPU op.
    Returns (unique pids in order of first appearance, idx_pad[P, Wmax] with -1 padding)."""
    pid = np.asarray(pid)
    assert len(pid) > 0, "no rows to pool"
    codes, uniques = pd.factorize(pid)
    pos = pd.Series(codes).groupby(codes).cumcount().to_numpy()
    idx_pad = np.full((len(uniques), int(pos.max()) + 1), -1, dtype=np.int64)
    idx_pad[codes, pos] = np.arange(len(codes))
    return np.asarray(uniques), idx_pad


def align_patients(g_pid, l_pid):
    """Fusion population = patients present in BOTH branches, sorted by pid (same as sorted(set & set))."""
    assert len(set(g_pid)) == len(g_pid) and len(set(l_pid)) == len(l_pid), "duplicate patient ids"
    common = np.intersect1d(g_pid, l_pid)
    return common, pd.Index(g_pid).get_indexer(common), pd.Index(l_pid).get_indexer(common)

In [0]:
# ============================================================================
# Model + losses (unchanged from the fusion notebook)
# ============================================================================
def class_offsets(prototypes_per_class):
    offsets, start = {}, 0
    for name in CLASS_NAMES:
        k = prototypes_per_class[name]
        offsets[name] = (start, start + k)
        start += k
    return offsets, start


class AdapterMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.BatchNorm1d(out_dim, affine=False),
        )
    def forward(self, x):
        return self.net(x)


class PrototypeLayer(nn.Module):
    def __init__(self, prototypes_per_class, prototype_dim, scale, verbose=True):
        super().__init__()
        self.prototypes_per_class = prototypes_per_class
        self.offsets, self.n_prototypes = class_offsets(prototypes_per_class)
        self.prototype_dim = prototype_dim
        self.scale = scale
        self.prototypes = nn.Parameter(torch.randn(self.n_prototypes, prototype_dim) * 0.1)
        assignment = torch.zeros(len(CLASS_NAMES), self.n_prototypes)
        for c, name in enumerate(CLASS_NAMES):
            s, e = self.offsets[name]
            assignment[c, s:e] = 1.0
        self.register_buffer("assignment", assignment)

    def forward(self, z):
        z_norm = F.normalize(z, dim=1)
        p_norm = F.normalize(self.prototypes, dim=1)
        return self.scale * (z_norm @ p_norm.T)


class ProtoEHRNet(nn.Module):
    def __init__(self, cfg, scale, verbose=True):
        super().__init__()
        self.adapter = AdapterMLP(cfg["embedding_dim"], cfg["adapter_hidden"], cfg["adapter_out"])
        self.proto_layer = PrototypeLayer(cfg["prototypes_per_class"], cfg["adapter_out"], scale, verbose)
        self.classifier = nn.Linear(self.proto_layer.n_prototypes, cfg["n_classes"], bias=True)

    def forward(self, x):
        z = self.adapter(x)
        sim = self.proto_layer(z)
        return self.classifier(sim), sim, z


def bce_loss(logits, labels, class_weights):
    return F.binary_cross_entropy_with_logits(logits, labels, pos_weight=class_weights, reduction="mean")

def clustering_loss(sim, labels, assignment):
    pos_mask = (labels @ assignment).clamp(0, 1).bool()
    masked = sim.masked_fill(~pos_mask, -1e9)
    has_pos = pos_mask.any(dim=1)
    return -masked[has_pos].max(dim=1).values.mean() if has_pos.any() else sim.new_tensor(0.0)

def separation_loss(sim, labels, assignment):
    neg_mask = ((labels @ assignment) == 0)
    masked = sim.masked_fill(~neg_mask, -1e9)
    has_neg = neg_mask.any(dim=1)
    return masked[has_neg].max(dim=1).values.mean() if has_neg.any() else sim.new_tensor(0.0)

def orthogonality_loss(prototypes):
    p = F.normalize(prototypes, dim=1)
    gram = p @ p.T
    I = torch.eye(gram.size(0), device=gram.device)
    return torch.norm(gram - I, p="fro") ** 2

def build_jaccard_matrix(Y_train, prototypes_per_class, zero_diagonal_blocks=True):
    n_classes = len(CLASS_NAMES)
    C_class = np.zeros((n_classes, n_classes))
    for c1 in range(n_classes):
        for c2 in range(n_classes):
            inter = ((Y_train[:, c1] == 1) & (Y_train[:, c2] == 1)).sum()
            union = ((Y_train[:, c1] == 1) | (Y_train[:, c2] == 1)).sum()
            C_class[c1, c2] = inter / union if union > 0 else 0.0
    if zero_diagonal_blocks:
        np.fill_diagonal(C_class, 0.0)
    offsets, P = class_offsets(prototypes_per_class)
    C_proto = np.zeros((P, P))
    for c1, n1 in enumerate(CLASS_NAMES):
        for c2, n2 in enumerate(CLASS_NAMES):
            rs, re = offsets[n1]; cs, ce = offsets[n2]
            C_proto[rs:re, cs:ce] = C_class[c1, c2]
    return torch.tensor(C_proto, dtype=torch.float32)

def contrastive_loss(prototypes, C):
    p = F.normalize(prototypes, dim=1)
    S = p @ p.T
    C_comp = 1.0 - C
    pos_term = (C * S).sum() / C.sum().clamp(min=1e-8)
    neg_term = (C_comp * S).sum() / C_comp.sum().clamp(min=1e-8)
    return -(1.0 / prototypes.size(0) ** 0.5) * (pos_term - neg_term)

def covariance_loss(z, eps=1e-4):
    z = z - z.mean(dim=0, keepdim=True)
    n, d = z.shape
    cov = (z.T @ z) / (n - 1)
    off_diag = cov - torch.diag(torch.diagonal(cov))
    return (off_diag ** 2).sum() / d


def prototype_diagnostics(model):
    offsets, _ = class_offsets(model.proto_layer.prototypes_per_class)
    with torch.no_grad():
        p = F.normalize(model.proto_layer.prototypes.cpu(), dim=1)
        gram = (p @ p.T).numpy()
    concentrations = []
    for name in CLASS_NAMES:
        s, e = offsets[name]; k = e - s
        if k <= 1:
            continue
        block = gram[s:e, s:e]
        concentrations.append((block.sum() - np.trace(block)) / (k * (k - 1)))
    return {"proto_concentration": float(np.mean(concentrations)) if concentrations else float("nan")}

In [0]:
# ============================================================================
# Metric suite (unchanged) + fast macro AUROC for per-epoch early stopping
# ============================================================================
def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
    results = {}
    y_pred = (y_prob >= 0.5).astype(int)
    for c, name in enumerate(class_names):
        yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
        n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
        results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"f1_{name}"] = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
        results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
        results[f"bacc_{name}"] = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

    def macro(prefix):
        vals = [results[f"{prefix}_{n}"] for n in class_names if not np.isnan(results[f"{prefix}_{n}"])]
        return float(np.mean(vals)) if vals else float("nan")

    for m in ["auroc", "auprc", "f1", "bacc", "brier"]:
        results[f"{m}_macro"] = macro(m)
    for m in ["auroc", "auprc"]:
        vals = [results[f"{m}_{n}"] for n in outcome_names if not np.isnan(results[f"{m}_{n}"])]
        results[f"{m}_macro_outcomes_only"] = float(np.mean(vals)) if vals else float("nan")
    results["_primary"] = results["auroc_macro"]
    return results


def macro_auroc(y_true, y_prob):
    """== _compute_metric_suite(...)['auroc_macro'] but computes only that (early stopping runs it every epoch)."""
    vals = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        n_pos = int(yt.sum())
        if n_pos > 0 and n_pos < len(yt):
            vals.append(roc_auc_score(yt, y_prob[:, c]))
    return float(np.mean(vals)) if vals else float("nan")

In [0]:
# ============================================================================
# GPU-resident batching / pooling / eval / projection / branch training
# ============================================================================
class GPUBatches:
    """Replaces DataLoader(TensorDataset, shuffle=True). All tensors already live on the GPU; each epoch draws a
    fresh randperm on the GPU (seeded via torch.manual_seed) and gathers batches by index: no worker processes,
    collate or host->device copies. Same math as the DataLoader; only the shuffle stream differs."""
    def __init__(self, *tensors, batch_size, drop_last):
        self.tensors, self.bs, self.drop_last = tensors, batch_size, drop_last
        self.n = len(tensors[0])
    def __len__(self):
        return self.n // self.bs if self.drop_last else -(-self.n // self.bs)
    def __iter__(self):
        perm = torch.randperm(self.n, device=self.tensors[0].device)
        for i in range(len(self)):
            idx = perm[i * self.bs:(i + 1) * self.bs]
            yield tuple(t[idx] for t in self.tensors)


class PatientPooler:
    """LOCAL-branch top-k pooling as one batched GPU op (replaces the pandas groupby-apply).
    For each patient and prototype: mean of the k largest row similarities (or all rows if fewer than k),
    identical to np.sort(vals)[-k_eff:].mean(). Patient order = first appearance (== groupby(sort=False))."""
    def __init__(self, pid, Y_rows, k, device):
        self.pids, idx_pad = build_pad_index(pid)
        self.idx_pad = torch.as_tensor(idx_pad, device=device)
        self.k = min(k, idx_pad.shape[1])
        self.Y = np.asarray(Y_rows)[idx_pad[:, 0]]          # labels are patient-level, so any row works
        self.n_rows = (idx_pad >= 0).sum(axis=1)
    @torch.no_grad()
    def pool(self, sim_rows):                                # (R, n_proto) -> (P, n_proto)
        NEG = -1e30
        ext = torch.cat([sim_rows, sim_rows.new_full((1, sim_rows.shape[1]), NEG)], dim=0)
        top = torch.topk(ext[self.idx_pad], self.k, dim=1).values   # -1 index -> the NEG pad row
        valid = top > NEG / 2
        return torch.where(valid, top, torch.zeros_like(top)).sum(1) / valid.sum(1).clamp(min=1)


@torch.no_grad()
def row_similarities(model, X_g, chunk=None):
    """Eval-mode similarities, chunked. BatchNorm uses running stats in eval, so chunk size cannot change results."""
    model.eval()
    chunk = chunk or CFG["infer_chunk"]
    return torch.cat([model.proto_layer(model.adapter(X_g[s:s + chunk])) for s in range(0, len(X_g), chunk)], 0)


@torch.no_grad()
def predict_probs(model, branch, X_g, pooler=None):
    sim = row_similarities(model, X_g)
    if branch == "local":
        sim = pooler.pool(sim)                               # classified once per patient, as before
    return torch.sigmoid(model.classifier(sim)).cpu().numpy()


@torch.no_grad()
def project_prototypes(model, X_train_g, Y_train, pid_train, device, verbose=True):
    """Same cross-class exclusion + soft per-slot comorbidity preference as the fusion notebook;
    only the encoding loop changed (GPU-resident X, larger chunks)."""
    model.eval()
    all_z = torch.cat([model.adapter(X_train_g[s:s + CFG["infer_chunk"]])
                       for s in range(0, len(X_train_g), CFG["infer_chunk"])], 0)
    z_norm = F.normalize(all_z, dim=1)
    p_norm_all = F.normalize(model.proto_layer.prototypes, dim=1)
    scale = model.proto_layer.scale
    used_mask = torch.zeros(len(X_train_g), dtype=torch.bool, device=device)
    purity_mask = torch.tensor(Y_train.sum(axis=1), device=device) == 1

    proj = {}
    for name in CLASS_NAMES:
        s, e = model.proto_layer.offsets[name]
        class_idx = CLASS_NAMES.index(name)
        class_mask = torch.tensor(Y_train[:, class_idx] == 1, device=device)
        if not class_mask.any():
            continue
        p_block = p_norm_all[s:e]
        for k in range(e - s):
            p_idx = s + k
            # prefer pure (single-label) unclaimed patients; fall back to any unclaimed same-class patient
            pure_eligible = class_mask & (~used_mask) & purity_mask
            eligible_mask = pure_eligible if pure_eligible.any() else (class_mask & (~used_mask))
            if not eligible_mask.any():
                if verbose:
                    print(f"  WARNING: class '{name}' ran out of eligible patients at slot {k+1}/{e-s} -- skipping slot.")
                continue
            eligible_idxs = torch.where(eligible_mask)[0]
            sims = scale * (p_block[k:k + 1] @ z_norm[eligible_mask].T).squeeze(0)
            best_local = sims.argmax().item()
            train_idx = int(eligible_idxs[best_local].item())
            model.proto_layer.prototypes[p_idx].copy_(all_z[train_idx])
            proj[p_idx] = {"train_idx": train_idx, "patient_id": str(pid_train[train_idx]),
                           "class_idx": class_idx, "class_name": name,
                           "sim_score": float(sims[best_local].item()),
                           "used_comorbid_fallback": not bool(purity_mask[train_idx].item())}
            used_mask[train_idx] = True
    if verbose:
        n_fb = sum(1 for v in proj.values() if v["used_comorbid_fallback"])
        print(f"  projected {len(proj)} prototypes ({n_fb} used the comorbid fallback)")
    return proj


def run_one_epoch(model, loader, optimizer, C_jac, class_weights, cfg):
    """Training epoch. Loss terms are identical to the fusion notebook; the per-epoch loss bookkeeping
    (never used by the caller) was dropped to avoid 7 extra device ops per step."""
    model.train(True)
    for X_b, Y_b in loader:
        logits, sim, z = model(X_b)
        l_total = (bce_loss(logits, Y_b, class_weights)
                   + cfg["lambda_clst"] * clustering_loss(sim, Y_b, model.proto_layer.assignment)
                   + cfg["lambda_sep"] * separation_loss(sim, Y_b, model.proto_layer.assignment)
                   + cfg["lambda_div"] * orthogonality_loss(model.proto_layer.prototypes)
                   + cfg["lambda_cntrst"] * contrastive_loss(model.proto_layer.prototypes, C_jac)
                   + cfg.get("lambda_covar", 0.0) * covariance_loss(z))
        optimizer.zero_grad(set_to_none=True)
        l_total.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()


def train_protoehr(cfg, data, branch, pooler_val=None, verbose=True):
    """Same training schedule / early stopping / re-projection logic as the fusion notebook's train_protoehr,
    with GPU-resident data, vectorized validation, and macro-AUROC-only per-epoch validation."""
    torch.manual_seed(cfg["random_seed"]); np.random.seed(cfg["random_seed"])
    Y_train, pid_train = data["Y_train"], data["pid_train"]
    class_weights = data["class_weights"]
    val_Y = data["Y_val"] if branch == "global" else pooler_val.Y

    loader = GPUBatches(data["X_train_g"], data["Y_train_g"], batch_size=cfg["batch_size"], drop_last=True)
    assert len(loader) > 0, "training set smaller than one batch"

    def val_probs(model):
        return predict_probs(model, branch, data["X_val_g"], pooler_val)

    scale = cfg["adapter_out"] ** 0.5
    model = ProtoEHRNet(cfg, scale, verbose=verbose).to(device)
    C_jac = build_jaccard_matrix(Y_train, cfg["prototypes_per_class"], cfg["zero_diagonal_blocks"]).to(device)

    best_iter_metric, iter_patience_ctr = -1.0, 0
    current_lr = cfg["lr"]
    overall_best_state, projection_map = None, {}

    for iteration in range(1, cfg["max_iterations"] + 1):
        optimizer = torch.optim.Adam(model.parameters(), lr=current_lr, weight_decay=cfg["weight_decay"])
        best_epoch_metric, epoch_patience, best_epoch_state = -1.0, 0, None

        for epoch in range(1, cfg["max_epochs"] + 1):
            run_one_epoch(model, loader, optimizer, C_jac, class_weights, cfg)
            primary = macro_auroc(val_Y, val_probs(model))
            improved = (not np.isnan(primary)) and (primary > best_epoch_metric)
            if improved or epoch == 1:
                if improved:
                    best_epoch_metric = primary; epoch_patience = 0
                best_epoch_state = {k: v.clone() for k, v in model.state_dict().items()}
                if not improved:
                    epoch_patience += 1
            else:
                epoch_patience += 1
            if epoch_patience >= cfg["patience"]:
                break

        model.load_state_dict(best_epoch_state)
        if best_epoch_metric > best_iter_metric:
            best_iter_metric = best_epoch_metric; iter_patience_ctr = 0
            overall_best_state = copy.deepcopy(best_epoch_state)
        else:
            iter_patience_ctr += 1

        projection_map = project_prototypes(model, data["X_train_g"], Y_train, pid_train, device, verbose=verbose)
        if verbose:
            print(f"  [{branch}] iter {iteration}/{cfg['max_iterations']} best={best_epoch_metric:.4f} "
                  f"all-time={best_iter_metric:.4f} iter_pat={iter_patience_ctr}/{cfg['iter_patience']} (ep {epoch})")
        if iter_patience_ctr >= cfg["iter_patience"]:
            model.load_state_dict(overall_best_state)
            projection_map = project_prototypes(model, data["X_train_g"], Y_train, pid_train, device, verbose=verbose)
            break
        current_lr *= cfg["lr_decay"]

    model.load_state_dict(overall_best_state)
    final_metrics = _compute_metric_suite(val_Y, val_probs(model))
    return model, final_metrics, prototype_diagnostics(model), projection_map

In [0]:
# ============================================================================
# Fusion (unchanged model/loss; GPU-resident batches) + bootstrap + result writers
# ============================================================================
def zscore_fit(X):
    return X.mean(axis=0, keepdims=True), X.std(axis=0, keepdims=True).clip(min=1e-8)

def zscore_apply(X, mean, std):
    return (X - mean) / std


class ConcatFusion(nn.Module):
    """Concatenation of BOTH branches' full similarity vectors + age covariate -> one linear layer."""
    def __init__(self, n_local_prototypes, n_global_prototypes, n_classes, n_covariates=1):
        super().__init__()
        self.classifier = nn.Linear(n_local_prototypes + n_global_prototypes + n_covariates, n_classes)
    def forward(self, sim_local, sim_global, covariates):
        return self.classifier(torch.cat([sim_local, sim_global, covariates], dim=1))


def train_fusion_layer(sim_l_tr, sim_g_tr, age_tr, Y_tr, sim_l_va, sim_g_va, age_va, Y_va,
                       n_local, n_global, n_classes, class_weights, device, fcfg, verbose=True):
    """Early stopping on val macro AUROC; L1 only on the classifier weight matrix (not the bias)."""
    torch.manual_seed(fcfg["seed"])        # the original relied on leftover global RNG state; seed explicitly
    model = ConcatFusion(n_local, n_global, n_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=fcfg["lr"], weight_decay=fcfg["weight_decay"])
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=class_weights)

    age_mean, age_std = age_tr.mean(), age_tr.std() + 1e-8
    T = lambda a: torch.as_tensor(a, dtype=torch.float32, device=device)
    xl, xg, a, y = T(sim_l_tr), T(sim_g_tr), T(((age_tr - age_mean) / age_std).reshape(-1, 1)), T(Y_tr)
    xl_v, xg_v, a_v = T(sim_l_va), T(sim_g_va), T(((age_va - age_mean) / age_std).reshape(-1, 1))
    loader = GPUBatches(xl, xg, a, y, batch_size=fcfg["batch_size"], drop_last=False)

    best_auroc, best_state, patience_ctr = -1.0, None, 0
    for epoch in range(1, fcfg["max_epochs"] + 1):
        model.train()
        for b_xl, b_xg, b_a, b_y in loader:
            loss = loss_fn(model(b_xl, b_xg, b_a), b_y)
            if fcfg["lambda_l1"] > 0:
                loss = loss + fcfg["lambda_l1"] * model.classifier.weight.abs().sum()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p_val = torch.sigmoid(model(xl_v, xg_v, a_v)).cpu().numpy()
        auroc = macro_auroc(Y_va, p_val)
        if auroc > best_auroc:
            best_auroc, patience_ctr = auroc, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(f"  fusion ep {epoch:3d} | val macro AUROC {auroc:.4f} | pat {patience_ctr}/{fcfg['patience']}")
        if patience_ctr >= fcfg["patience"]:
            break
    if best_state is None:
        raise RuntimeError("fusion never produced a finite validation AUROC")
    model.load_state_dict(best_state)
    return model, best_auroc, age_mean, age_std


def bootstrap_metrics_ci(Y_true, Y_prob, class_names, n_bootstrap, confidence, seed):
    """Same procedure as the fusion notebook: resample test patients, recompute the whole suite, percentile CI."""
    rng = np.random.RandomState(seed)
    n = Y_true.shape[0]
    stems = ["auroc", "auprc", "f1", "bacc", "brier"]
    keys = ([f"{m}_macro" for m in stems] + ["auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]
            + [f"{m}_{c}" for m in stems for c in class_names])
    boot = {k: [] for k in keys}
    n_skipped = 0
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        yt_r, yp_r = Y_true[idx], Y_prob[idx]
        if any(yt_r[:, c].sum() == 0 or yt_r[:, c].sum() == n for c in range(len(class_names))):
            n_skipped += 1
            continue
        m = _compute_metric_suite(yt_r, yp_r, class_names=class_names, outcome_names=OUTCOME_NAMES)
        for k in keys:
            boot[k].append(m[k])
    res = {}
    for k in keys:
        arr = np.array(boot[k]); arr = arr[~np.isnan(arr)]
        if len(arr) == 0:
            res[k] = (float("nan"),) * 3
        else:
            lo, hi = np.percentile(arr, [(1 - confidence) / 2 * 100, (1 + confidence) / 2 * 100])
            res[k] = (arr.mean(), lo, hi)
    if n_skipped:
        print(f"  ({n_skipped}/{n_bootstrap} resamples skipped -- degenerate label in that draw)")
    return res


# ---- result rows ----
STEMS = ["auroc", "auprc", "f1", "bacc", "brier"]
_f = lambda v: None if (v is None or not np.isfinite(v)) else float(v)

def build_metric_rows(model_id, week, test_metrics, ci, n_test):
    rows = []
    def add(scope, cls, stem, key):
        _, lo, hi = ci[key]
        rows.append((model_id, week, scope, cls, stem, _f(test_metrics[key]), _f(lo), _f(hi), int(n_test)))
    for stem in STEMS:
        add("macro", "ALL", stem, f"{stem}_macro")
        for c in CLASS_NAMES:
            add("class", c, stem, f"{stem}_{c}")
    for stem in ["auroc", "auprc"]:
        add("macro_outcomes_only", "ALL", stem, f"{stem}_macro_outcomes_only")
    return rows

def size_row(model_id, week, split, population, Y):
    return (model_id, week, split, population, int(len(Y)), int(Y[:, 0].sum()), int(Y[:, 1].sum()), int(Y[:, 2].sum()))

METRICS_SCHEMA = StructType([StructField(n, t) for n, t in [
    ("model_id", StringType()), ("cutoff_week", IntegerType()), ("scope", StringType()), ("class_name", StringType()),
    ("metric", StringType()), ("estimate", DoubleType()), ("ci_lo", DoubleType()), ("ci_hi", DoubleType()),
    ("n_test", IntegerType())]])
SIZES_SCHEMA = StructType([StructField(n, t) for n, t in [
    ("model_id", StringType()), ("cutoff_week", IntegerType()), ("split", StringType()), ("population", StringType()),
    ("n_patients", IntegerType()), ("n_control", IntegerType()), ("n_pe_ghtn", IntegerType()), ("n_gdm", IntegerType())]])

def replace_model_rows(table, rows, schema, model_id):
    """Delete-then-append this model_id's rows (idempotent). Tables are partitioned by model_id so concurrent
    workers touch disjoint partitions; retries cover the occasional Delta commit conflict."""
    sdf = spark.createDataFrame(rows, schema).withColumn("run_ts", sf.current_timestamp())
    for attempt in range(CFG["write_retries"]):
        try:
            spark.sql(f"DELETE FROM {table} WHERE model_id = '{model_id}'")
            sdf.write.format("delta").mode("append").saveAsTable(table)
            return
        except Exception as e:
            if attempt == CFG["write_retries"] - 1:
                raise
            print(f"  write to {table} failed ({type(e).__name__}); retrying"); time.sleep(5 * (attempt + 1))


# ---- export helpers (same payload as the fusion notebook's export cell) ----
OMOP_TABLES = {
    "condition": ("catalog.schema.condition_occurrence_cohort", "condition_concept_id", "condition_source_concept_id"),
    "procedure": ("catalog.schema.procedure_occurrence_cohort", "procedure_concept_id", "procedure_source_concept_id"),
    "drug": ("catalog.schema.drug_exposure_cohort", "drug_concept_id", "drug_concept_id"),
    "measurement": ("catalog.schema.measurement_cohort", "measurement_concept_id", "measurement_source_concept_id"),
}

def fetch_code_sets_batched(person_ids):
    codes_by_patient = {str(p): set() for p in person_ids}
    id_list_sql = ", ".join(f"'{p}'" for p in person_ids)
    for domain, (table, concept_col, source_col) in OMOP_TABLES.items():
        df = spark.sql(f"""
            SELECT person_id, {concept_col} AS concept_id, {source_col} AS source_concept_id
            FROM {table} WHERE person_id IN ({id_list_sql})
        """).toPandas()
        if df.empty:
            continue
        cid = df["concept_id"].fillna(0).astype(int)
        sid = df["source_concept_id"].fillna(0).astype(int)
        df["resolved_id"] = cid.where(cid != 0, sid)
        df = df[df["resolved_id"] != 0]
        for pid, rid in zip(df["person_id"].astype(str), df["resolved_id"]):
            if pid in codes_by_patient:
                codes_by_patient[pid].add((domain, int(rid)))
    return codes_by_patient

In [0]:
# ============================================================================
# RUN: load -> train global -> train local -> fuse -> test eval -> write tables -> export
# ============================================================================
T0 = time.time()
tick = lambda msg: print(f"[{time.time()-T0:7.1f}s] {msg}")

# ---- 1. eligible patients + embeddings ----
coh = load_eligible_cohort(CUTOFF_WEEK)
assert coh["age"].notna().all(), f"{int(coh['age'].isna().sum())} eligible patients have NaN age"
tick(f"eligible at week {CUTOFF_WEEK}: {coh['split'].value_counts().to_dict()}")

global_data = attach_gpu(to_split_arrays(pull_embeddings(CFG["global_emb_table"], coh, local=False), coh, "global"))
local_data = attach_gpu(to_split_arrays(pull_embeddings(CFG["local_emb_table"], coh, local=True), coh, "local"))
n_g_rows = {s: len(global_data["X_" + s]) for s in ["train", "val", "test"]}
n_l_rows = {s: len(local_data["X_" + s]) for s in ["train", "val", "test"]}
tick(f"global rows: {n_g_rows} | local rows: {n_l_rows}")

poolers = {s: PatientPooler(local_data[f"pid_{s}"], local_data[f"Y_{s}"], CFG["topk_pool_k"], device)
           for s in ["train", "val", "test"]}
for s in ["train", "val", "test"]:
    assert len(set(global_data[f"pid_{s}"])) == len(global_data[f"pid_{s}"]), f"duplicate global patients in {s}"

# ---- 2. branches ----
global_model, global_val, global_diag, global_projection_map = train_protoehr(FINAL_CFG_GLOBAL, global_data, "global")
tick(f"global done: val macro AUROC {global_val['auroc_macro']:.4f}")
local_model, local_val, local_diag, local_projection_map = train_protoehr(
    FINAL_CFG_LOCAL, local_data, "local", pooler_val=poolers["val"])
tick(f"local done: val macro AUROC {local_val['auroc_macro']:.4f}")
for m_ in (global_model, local_model):
    for p_ in m_.parameters():
        p_.requires_grad_(False)
    m_.eval()
# record which window each local prototype came from (extra info for the cards)
for v in local_projection_map.values():
    v["window_name"] = str(local_data["window_name_train"][v["train_idx"]])

# ---- 3. fusion population = patients with a global embedding AND >=1 local row ----
def split_similarities(split):
    g_sim = row_similarities(global_model, global_data[f"X_{split}_g"]).cpu().numpy()
    l_sim = poolers[split].pool(row_similarities(local_model, local_data[f"X_{split}_g"])).cpu().numpy()
    return g_sim, l_sim

fusion = {}
sims = {}
for s in ["train", "val", "test"]:
    g_sim, l_sim = split_similarities(s)
    sims[s] = (g_sim, l_sim)
    common, g_rows, l_rows = align_patients(global_data[f"pid_{s}"], poolers[s].pids)
    fusion[s] = {"pids": common, "g": g_sim[g_rows], "l": l_sim[l_rows],
                 "Y": global_data[f"Y_{s}"][g_rows], "age": global_data[f"age_{s}"][g_rows]}
n_fusion = {s: len(fusion[s]["pids"]) for s in fusion}
tick(f"fusion patients: {n_fusion}")

l_mean, l_std = zscore_fit(fusion["train"]["l"]); g_mean, g_std = zscore_fit(fusion["train"]["g"])
Z = {s: (zscore_apply(fusion[s]["l"], l_mean, l_std), zscore_apply(fusion[s]["g"], g_mean, g_std)) for s in fusion}
n_local_prototypes, n_global_prototypes = Z["train"][0].shape[1], Z["train"][1].shape[1]
pos_freq = fusion["train"]["Y"].mean(axis=0).clip(0.03, 0.97)
fusion_class_weights = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float32, device=device)

fusion_model, fusion_val_auroc, age_mean, age_std = train_fusion_layer(
    Z["train"][0], Z["train"][1], fusion["train"]["age"], fusion["train"]["Y"],
    Z["val"][0], Z["val"][1], fusion["val"]["age"], fusion["val"]["Y"],
    n_local_prototypes, n_global_prototypes, len(CLASS_NAMES), fusion_class_weights, device, CFG["fusion"])
tick(f"fusion done: val macro AUROC {fusion_val_auroc:.4f}")

# ---- 4. FINAL held-out test evaluation (touched once) ----
pids_test, Y_fusion_test, age_test = fusion["test"]["pids"], fusion["test"]["Y"], fusion["test"]["age"]
with torch.no_grad():
    T_ = lambda a: torch.as_tensor(a, dtype=torch.float32, device=device)
    p_test = torch.sigmoid(fusion_model(T_(Z["test"][0]), T_(Z["test"][1]),
                                        T_(((age_test - age_mean) / age_std).reshape(-1, 1)))).cpu().numpy()
test_metrics = _compute_metric_suite(Y_fusion_test, p_test)
print("=== HELD-OUT TEST (fused) ===")
for k in ["auroc_macro", "auprc_macro", "f1_macro", "bacc_macro", "brier_macro",
          "auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]:
    print(f"  {k:<28} {test_metrics[k]:.4f}")
for c in CLASS_NAMES:
    print(f"  {c:8s} AUROC {test_metrics[f'auroc_{c}']:.4f}  AUPRC {test_metrics[f'auprc_{c}']:.4f}  "
          f"F1 {test_metrics[f'f1_{c}']:.4f}  BalAcc {test_metrics[f'bacc_{c}']:.4f}  Brier {test_metrics[f'brier_{c}']:.4f}")

test_ci = bootstrap_metrics_ci(Y_fusion_test, p_test, CLASS_NAMES, CFG["n_bootstrap"], CFG["ci"], CFG["bootstrap_seed"])
tick("bootstrap done")

# ---- 5. Delta tables ----
mid, wk = CFG["model_id"], CUTOFF_WEEK
replace_model_rows(CFG["metrics_table"], build_metric_rows(mid, wk, test_metrics, test_ci, len(pids_test)),
                   METRICS_SCHEMA, mid)
size_rows = [size_row(mid, wk, s, "eligible", coh.loc[coh["split"] == s, LABEL_COLS].to_numpy()) for s in ["train", "val", "test"]]
size_rows += [size_row(mid, wk, s, "global", global_data[f"Y_{s}"]) for s in ["train", "val", "test"]]
size_rows += [size_row(mid, wk, s, "local", poolers[s].Y) for s in ["train", "val", "test"]]
size_rows += [size_row(mid, wk, s, "fusion", fusion[s]["Y"]) for s in ["train", "val", "test"]]
replace_model_rows(CFG["sizes_table"], size_rows, SIZES_SCHEMA, mid)
tick("Delta tables written")

In [0]:
# ============================================================================
# Pickle export -- same keys as the fusion notebook's export (+ model_id / cutoff / cfg / CIs)
# ============================================================================
g_idx = sorted(global_projection_map.keys()); l_idx = sorted(local_projection_map.keys())

# similarity matrices over ALL test patients of each branch (as in the original export)
g_sim_test_all, l_sim_test_all = sims["test"]
assert g_sim_test_all.shape[1] == len(g_idx) and l_sim_test_all.shape[1] == len(l_idx), \
    "a prototype slot was skipped during projection; similarity columns no longer match projection_map keys"
global_similarity_df = pd.DataFrame(g_sim_test_all, index=global_data["pid_test"], columns=g_idx)
local_similarity_df = pd.DataFrame(l_sim_test_all, index=poolers["test"].pids, columns=l_idx)

# code sets: test patients come from the one-time cache; prototype patients (~70) are fetched here
with open(CFG["test_code_set_cache"], "rb") as f:
    _cache = pickle.load(f)
test_patient_code_sets = {p: _cache[p] for p in pids_test}
all_prototype_patient_ids = list(set([i["patient_id"] for i in global_projection_map.values()]
                                     + [i["patient_id"] for i in local_projection_map.values()]))
proto_raw = fetch_code_sets_batched(all_prototype_patient_ids)
global_prototype_code_sets = {p: proto_raw[i["patient_id"]] for p, i in global_projection_map.items()}
local_prototype_code_sets = {p: proto_raw[i["patient_id"]] for p, i in local_projection_map.items()}

g_train_idx = np.array([i["train_idx"] for i in global_projection_map.values()])
l_train_idx = np.array([i["train_idx"] for i in local_projection_map.values()])

export_payload = {
    "model_id": CFG["model_id"], "cutoff_week": CUTOFF_WEEK,
    "global_projection_map": global_projection_map, "local_projection_map": local_projection_map,
    "global_similarity_df": global_similarity_df, "local_similarity_df": local_similarity_df,
    "test_patient_code_sets": test_patient_code_sets,
    "global_prototype_code_sets": global_prototype_code_sets, "local_prototype_code_sets": local_prototype_code_sets,
    "all_prototype_patient_ids": all_prototype_patient_ids,
    "pids_test": pids_test, "Y_fusion_test": Y_fusion_test, "age_test": age_test, "p_test": p_test,
    "test_metrics": test_metrics, "test_metrics_ci": test_ci,
    "CLASS_NAMES": CLASS_NAMES,
    "n_local_prototypes": n_local_prototypes, "n_global_prototypes": n_global_prototypes,
    "fusion_classifier_weight": fusion_model.classifier.weight.detach().cpu().numpy(),
    "fusion_classifier_bias": fusion_model.classifier.bias.detach().cpu().numpy(),
    "ablation_results": None,      # ablations are not run in the landmark pipeline
    "global_X_test": global_data["X_test"], "global_pid_test": global_data["pid_test"],
    "global_proto_X": global_data["X_train"][g_train_idx], "global_proto_indices": g_idx,
    "local_X_test": local_data["X_test"], "local_pid_test": local_data["pid_test"],
    "local_proto_X": local_data["X_train"][l_train_idx], "local_proto_indices": l_idx,
    "global_adapter_state_dict": {k: v.cpu() for k, v in global_model.adapter.state_dict().items()},
    "global_proto_layer_state_dict": {k: v.cpu() for k, v in global_model.proto_layer.state_dict().items()},
    "global_adapter_cfg": {k: FINAL_CFG_GLOBAL[k] for k in ["embedding_dim", "adapter_hidden", "adapter_out"]},
    "local_adapter_state_dict": {k: v.cpu() for k, v in local_model.adapter.state_dict().items()},
    "local_proto_layer_state_dict": {k: v.cpu() for k, v in local_model.proto_layer.state_dict().items()},
    "local_adapter_cfg": {k: FINAL_CFG_LOCAL[k] for k in ["embedding_dim", "adapter_hidden", "adapter_out"]},
    "cfg_global": FINAL_CFG_GLOBAL, "cfg_local": FINAL_CFG_LOCAL, "cfg_fusion": CFG["fusion"],
    "z_score_stats": {"l_mean": l_mean, "l_std": l_std, "g_mean": g_mean, "g_std": g_std,
                      "age_mean": float(age_mean), "age_std": float(age_std)},
}
_pid0 = global_data["pid_test"][0]
export_payload["verification_check"] = {"patient_id": _pid0, "prototype_idx": g_idx[0],
                                        "expected_similarity": float(global_similarity_df.loc[_pid0, g_idx[0]])}

with open(CFG["export_path"], "wb") as f:
    pickle.dump(export_payload, f)
tick(f"exported {CFG['export_path']} ({os.path.getsize(CFG['export_path'])/1e6:.1f} MB)")

del global_data, local_data, export_payload
gc.collect(); torch.cuda.empty_cache()

In [0]:
%sql
SELECT model_id, cutoff_week, COUNT(*) AS n_rows
FROM catalog.schema.protopregnet_landmark_metrics
WHERE cutoff_week = 25
GROUP BY model_id, cutoff_week

In [0]:
%sql
DELETE FROM catalog.schema.protopregnet_landmark_metrics WHERE model_id = 'replication_v5';
DELETE FROM catalog.schema.protopregnet_landmark_sizes   WHERE model_id = 'replication_v5';